# NB04: Report & Figures

**Chay: Internet OFF, GPU OFF (CPU la du)**

Input: `results_all.csv` tu NB02/NB03 (Add lam Dataset)

Output:
- 7 Figures chuan paper (PNG 300 DPI)
- LaTeX Table 1 + Table 2 (paste thang vao .tex)
- Statistical significance test (t-test CNN vs Transformer)

In [ ]:
# ============================================================
# CONFIG
# ============================================================
import os
IS_KAGGLE = os.path.exists('/kaggle')
from pathlib import Path

# >>> SUA TEN DATASET (output cua NB03) <<<
RESULTS_DATASET = 'forest-eval-results'

RESULTS_DIR = (Path(f'/kaggle/input/{RESULTS_DATASET}') if IS_KAGGLE
               else Path('outputs/eval_results'))
OUT_DIR     = (Path('/kaggle/working/paper_figures') if IS_KAGGLE
               else Path('outputs/paper_figures'))
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ===========================================================
# Group mapping — KHONG co conflict (D dung config_id rieng)
# config format: <model>_<encoder>
# ===========================================================
GROUP_MAP = {
    # -- Group A: CNN Baseline --
    'unet_resnet34':               'A. CNN Baseline',
    'unet_resnet101':              'A. CNN Baseline',
    'deeplabv3plus_resnet101':     'A. CNN Baseline',
    'unetpp_resnet50':             'A. CNN Baseline',
    # -- Group B: Modern CNN --
    'unet_efficientnet-b5':                             'B. Modern CNN',
    'upernet_tu-convnext_base':                         'B. Modern CNN',
    'upernet_tu-convnext_large':                        'B. Modern CNN',
    # -- Group C: Transformer (native configs) --
    'segformer_mit_b5':                                 'C. Transformer',
    'deeplabv3plus_mit_b5':                          'C. Transformer',
    'upernet_tu-swinv2_base_window12to24_192to384':     'C. Transformer',
    'upernet_tu-swinv2_large_window12to24_192to384':    'C. Transformer',
    # -- Group D: Ablation (MiT-B3, 4 decoders) --
    # Note: segformer_mit_b3 belongs to D (ablation), NOT C
    'unet_mit_b3':            'D. Ablation',
    'deeplabv3plus_mit_b3':   'D. Ablation',
    'segformer_mit_b3':       'D. Ablation',
    'upernet_mit_b3':         'D. Ablation',
}
# Note: segformer_mit_b3 is intentionally ONLY in D (ablation group)
# If you also run it as a standalone C experiment, give it a different
# config name (e.g. segformer_mit_b3_standalone) in NB02

GROUP_COLORS = {
    'A. CNN Baseline': '#4878CF',
    'B. Modern CNN':   '#6ACC65',
    'C. Transformer':  '#D65F5F',
    'D. Ablation':     '#B47CC7',
}

CLASS_NAMES_SHORT = ['Sky','Deciduous','Coniferous','Fallen','Dirt',
                     'GndVeg','Rocks','Building','Fence','Car','Empty']
CLASS_NAMES_FULL  = ['Sky','Deciduous_trees','Coniferous_trees','Fallen_trees','Dirt_ground',
                     'Ground_vegetation','Rocks','Building','Fence','Car','Empty']

print(f'RESULTS_DIR: {RESULTS_DIR}')
print(f'OUT_DIR:     {OUT_DIR}')

In [ ]:
# ============================================================
# SETUP & LOAD DATA
# ============================================================
!pip install -q seaborn scipy

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

plt.rcParams.update({
    'font.family':    'DejaVu Sans',
    'font.size':      11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'xtick.labelsize':10,
    'ytick.labelsize':10,
    'figure.dpi':     150,
    'savefig.dpi':    300,
    'savefig.bbox':   'tight',
})

csv_path = RESULTS_DIR / 'results_all.csv'
if not csv_path.exists():
    raise FileNotFoundError(f'{csv_path} not found. Chay NB03 truoc!')

df_raw = pd.read_csv(csv_path)
print(f'Loaded: {len(df_raw)} rows | '
      f'{df_raw["config"].nunique()} configs | '
      f'{df_raw["seed"].nunique()} seeds')

# Add group labels
df_raw['group'] = df_raw['config'].map(GROUP_MAP).fillna('Other')
unmapped = df_raw[df_raw['group'] == 'Other']['config'].unique()
if len(unmapped) > 0:
    print(f'[WARN] Unmapped configs (will show as Other): {unmapped}')

# Aggregate: mean & std over seeds
iou_cols = [f'iou_{c}' for c in CLASS_NAMES_FULL]
num_cols  = ['mIoU', 'PA', 'mF1', 'params_M', 'flops_G', 'fps'] + iou_cols
mean_df   = df_raw.groupby('config')[num_cols].mean().round(2)
std_df    = df_raw.groupby('config')[num_cols].std().round(2).fillna(0)
mean_df['group'] = mean_df.index.map(
    lambda x: df_raw.loc[df_raw['config'] == x, 'group'].iloc[0])
mean_df = mean_df.sort_values('mIoU', ascending=False)

print('\n=== Top 5 by mIoU ===')
print(mean_df[['group', 'mIoU', 'PA', 'mF1', 'params_M', 'fps']].head().to_string())

In [ ]:
# ============================================================
# FIG 1: mIoU Comparison Bar Chart (all configs, sorted)
# ============================================================
fig, ax = plt.subplots(figsize=(14, max(6, len(mean_df) * 0.45)))

configs = mean_df.index.tolist()
miou    = mean_df['mIoU'].values
err     = std_df.loc[mean_df.index, 'mIoU'].values
colors  = [GROUP_COLORS.get(mean_df.loc[c, 'group'], '#999') for c in configs]

ax.barh(range(len(configs)), miou, xerr=err, color=colors,
        capsize=4, edgecolor='white', linewidth=0.5, height=0.7)

for i, (v, e) in enumerate(zip(miou, err)):
    ax.text(v + e + 0.3, i, f'{v:.1f}', va='center', fontsize=9, fontweight='bold')

ax.set_yticks(range(len(configs)))
ax.set_yticklabels(configs, fontsize=9)
ax.set_xlabel('mIoU (%)')
ax.set_title('Semantic Segmentation Comparison — Forest Inspection Dataset (Test: seq9)')
ax.set_xlim(0, max(miou) + 7)
ax.grid(axis='x', alpha=0.3)

handles = [mpatches.Patch(color=c, label=g) for g, c in GROUP_COLORS.items()]
ax.legend(handles=handles, loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig1_miou_comparison.png')
plt.show()
print('Saved: fig1_miou_comparison.png')

In [ ]:
# ============================================================
# FIG 2: Accuracy vs Complexity Scatter (FLOPs vs mIoU)
# Bubble size = number of parameters
# ============================================================
fig, ax = plt.subplots(figsize=(11, 7))

for group, gdf in mean_df.groupby('group'):
    ax.scatter(
        gdf['flops_G'], gdf['mIoU'],
        s=gdf['params_M'].clip(upper=500) * 2.5,
        c=GROUP_COLORS.get(group, '#999'),
        alpha=0.8, edgecolors='white', linewidth=0.8,
        label=group, zorder=3
    )
    for cfg, row in gdf.iterrows():
        short = (cfg.replace('upernet_tu-', '')
                    .replace('deeplabv3plus_', 'dlv3+_')
                    .replace('segformer_', 'sf_'))
        ax.annotate(short, (row['flops_G'], row['mIoU']),
                    textcoords='offset points', xytext=(5, 4), fontsize=7, alpha=0.85)

ax.set_xlabel('FLOPs (G)')
ax.set_ylabel('mIoU (%)')
ax.set_title('Accuracy vs. Computational Complexity\n(bubble size ∝ number of parameters)')

# Bubble size legend
for sz in [20, 100, 300]:
    ax.scatter([], [], s=sz*2.5, c='#ccc', edgecolors='#888', label=f'{sz}M params')

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, loc='lower right', fontsize=9, title='Legend')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig2_accuracy_complexity.png')
plt.show()
print('Saved: fig2_accuracy_complexity.png')

In [ ]:
# ============================================================
# FIG 3: Per-class IoU Heatmap (all configs)
# ============================================================
iou_mat = mean_df[[f'iou_{c}' for c in CLASS_NAMES_FULL]].copy()
iou_mat.columns = CLASS_NAMES_SHORT
iou_mat = iou_mat.replace(-1, np.nan)   # -1 = class not present

fig, ax = plt.subplots(figsize=(14, max(8, len(iou_mat) * 0.55)))
sns.heatmap(iou_mat.astype(float), annot=True, fmt='.1f',
            cmap='RdYlGn', linewidths=0.4, ax=ax,
            vmin=0, vmax=100, cbar_kws={'label': 'IoU (%)'})
ax.set_title('Per-class IoU (%) — All Configurations')
ax.set_xlabel('Class')
ax.set_ylabel('Model Configuration')
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig3_per_class_iou_heatmap.png')
plt.show()
print('Saved: fig3_per_class_iou_heatmap.png')

In [ ]:
# ============================================================
# FIG 4: Ablation Study — Group D (fixed encoder MiT-B3)
# ============================================================
ablation_df = mean_df[mean_df['group'] == 'D. Ablation'].copy()

if len(ablation_df) > 0:
    # Decoder label: take first token of config name
    decoder_map = {
        'unet':           'UNet',
        'deeplabv3plus':  'DeepLabV3+',
        'segformer':      'SegFormer',
        'upernet':        'UPerNet',
    }
    labels_ab = [decoder_map.get(c.split('_')[0], c.split('_')[0]) for c in ablation_df.index]
    miou_ab   = ablation_df['mIoU'].values
    err_ab    = std_df.loc[ablation_df.index, 'mIoU'].values

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Left: mIoU bar
    bars = axes[0].bar(labels_ab, miou_ab, yerr=err_ab,
                       color=GROUP_COLORS['D. Ablation'],
                       capsize=5, edgecolor='white', width=0.6)
    axes[0].set_ylabel('mIoU (%)')
    axes[0].set_title('Decoder Impact (Encoder: MiT-B3)')
    axes[0].set_ylim(0, max(miou_ab) * 1.18)
    axes[0].grid(axis='y', alpha=0.3)
    for bar, val, std_val in zip(bars, miou_ab, err_ab):
        axes[0].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + std_val + 0.4,
                     f'{val:.1f}%', ha='center', va='bottom',
                     fontsize=11, fontweight='bold')

    # Right: grouped bar mIoU / PA / mF1
    met_cols   = ['mIoU', 'PA', 'mF1']
    met_colors = ['#D65F5F', '#4878CF', '#6ACC65']
    x = np.arange(len(labels_ab))
    w = 0.25
    for j, (met, col) in enumerate(zip(met_cols, met_colors)):
        axes[1].bar(x + j * w, ablation_df[met].values, w,
                    label=met, color=col, alpha=0.85, edgecolor='white')
    axes[1].set_xticks(x + w)
    axes[1].set_xticklabels(labels_ab)
    axes[1].set_ylabel('Score (%)')
    axes[1].set_title('Multi-metric (Encoder: MiT-B3)')
    axes[1].legend(fontsize=10)
    axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle('Ablation Study: Decoder Architecture Impact', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'fig4_ablation_decoder.png')
    plt.show()
    print('Saved: fig4_ablation_decoder.png')
else:
    print('[SKIP] No Group D data')

In [ ]:
# ============================================================
# FIG 5: Box Plot mIoU by Group
# ============================================================
groups_order = ['A. CNN Baseline', 'B. Modern CNN', 'C. Transformer']
plot_data    = [df_raw[df_raw['group'] == g]['mIoU'].values for g in groups_order]
plot_data    = [d for d in plot_data if len(d) > 0]   # skip empty groups
labels_bp    = [g for g, d in zip(groups_order, [df_raw[df_raw['group']==g]['mIoU'].values for g in groups_order]) if len(d) > 0]

fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(plot_data, labels=labels_bp, patch_artist=True,
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(linewidth=1.5),
                capprops=dict(linewidth=2))
for patch, lbl in zip(bp['boxes'], labels_bp):
    patch.set_facecolor(GROUP_COLORS[lbl])
    patch.set_alpha(0.75)

np.random.seed(42)
for i, data in enumerate(plot_data, 1):
    jitter = np.random.uniform(-0.1, 0.1, len(data))
    ax.scatter(np.full_like(data, float(i)) + jitter, data,
               zorder=3, color='black', alpha=0.6, s=30)

ax.set_ylabel('mIoU (%)')
ax.set_title('mIoU Distribution by Architecture Group\n(each point = 1 run)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig5_boxplot_groups.png')
plt.show()
print('Saved: fig5_boxplot_groups.png')

In [ ]:
# ============================================================
# FIG 6: Radar Chart — Top 5 Models
# ============================================================
top5      = mean_df.head(5).copy()
top5_plot = top5[['mIoU', 'PA', 'mF1']].copy()

# Normalize FPS and Params to 0-100
fps_min, fps_max = top5['fps'].min(), top5['fps'].max()
par_min, par_max = top5['params_M'].min(), top5['params_M'].max()
top5_plot['Speed(norm)']   = ((top5['fps'] - fps_min) / (fps_max - fps_min + 1e-6) * 100).values
top5_plot['Compact(norm)'] = (100 - (top5['params_M'] - par_min) / (par_max - par_min + 1e-6) * 100).values

labels_r = top5_plot.columns.tolist()
N        = len(labels_r)
angles   = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors_r = plt.cm.Set2(np.linspace(0, 1, len(top5)))

for idx, (cfg, row) in enumerate(top5_plot.iterrows()):
    vals = row.values.tolist() + [row.values[0]]
    ax.plot(angles, vals, 'o-', linewidth=2, label=cfg[:28], color=colors_r[idx])
    ax.fill(angles, vals, alpha=0.08, color=colors_r[idx])

ax.set_thetagrids(np.degrees(angles[:-1]), labels_r)
ax.set_ylim(0, 100)
ax.set_title('Multi-metric Radar — Top 5 Models', y=1.08, fontsize=13)
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig6_radar_top5.png')
plt.show()
print('Saved: fig6_radar_top5.png')

In [ ]:
# ============================================================
# FIG 7: Per-class IoU bar — Top 3 Models
# ============================================================
top3         = mean_df.head(3)
iou_col_full = [f'iou_{c}' for c in CLASS_NAMES_FULL]

fig, ax = plt.subplots(figsize=(14, 5))
x           = np.arange(len(CLASS_NAMES_SHORT))
w           = 0.28
colors_top3 = ['#D65F5F', '#4878CF', '#6ACC65']

for i, (cfg, row) in enumerate(top3.iterrows()):
    vals = [max(0, row[c]) for c in iou_col_full]   # -1 -> 0
    ax.bar(x + i * w, vals, w, label=cfg, color=colors_top3[i],
           alpha=0.85, edgecolor='white')

ax.set_xticks(x + w)
ax.set_xticklabels(CLASS_NAMES_SHORT, rotation=25, ha='right')
ax.set_ylabel('IoU (%)')
ax.set_title('Per-class IoU Comparison — Top 3 Models')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig7_per_class_top3.png')
plt.show()
print('Saved: fig7_per_class_top3.png')

In [ ]:
# ============================================================
# STATISTICAL SIGNIFICANCE TEST (Welch t-test)
# ============================================================
print('=== Statistical Significance Test (Welch t-test) ===')
print('H0: No difference in mIoU between CNN and Transformer groups')
print('alpha = 0.05\n')

cnn_data  = df_raw[df_raw['group'].isin(['A. CNN Baseline', 'B. Modern CNN'])]['mIoU'].values
trans_data= df_raw[df_raw['group'] == 'C. Transformer']['mIoU'].values

if len(cnn_data) > 1 and len(trans_data) > 1:
    # Welch t-test (does not assume equal variance)
    t_stat, p_val = stats.ttest_ind(trans_data, cnn_data, equal_var=False)
    print(f'CNN (A+B):       n={len(cnn_data)},  mean={cnn_data.mean():.2f}%, std={cnn_data.std():.2f}%')
    print(f'Transformer (C): n={len(trans_data)}, mean={trans_data.mean():.2f}%, std={trans_data.std():.2f}%')
    print(f'\nt-statistic = {t_stat:.4f}')
    print(f'p-value     = {p_val:.4f}')

    if p_val < 0.01:
        sig = '** (p < 0.01, highly significant)'
    elif p_val < 0.05:
        sig = '* (p < 0.05, significant)'
    else:
        sig = 'NS (not significant at alpha=0.05)'
    print(f'\nResult: {sig}')

    # Cohen's d effect size
    pooled_std = np.sqrt((trans_data.std()**2 + cnn_data.std()**2) / 2)
    d = (trans_data.mean() - cnn_data.mean()) / (pooled_std + 1e-9)
    effect = 'large' if abs(d) > 0.8 else 'medium' if abs(d) > 0.5 else 'small'
    print(f"Cohen's d = {d:.3f} ({effect} effect)")

    print()
    print('Suggested paper sentence:')
    direction = 'outperform' if trans_data.mean() > cnn_data.mean() else 'underperform'
    print(f'"Transformer-based models (Group C) {direction} CNN-based models')
    print(f' (Group A+B) by {abs(trans_data.mean()-cnn_data.mean()):.1f}% mIoU on average')
    print(f' (Welch t-test, t={t_stat:.2f}, p={p_val:.3f}, d={d:.2f})."')
else:
    print('[SKIP] Not enough data for t-test')

In [ ]:
# ============================================================
# LATEX TABLE 1: Main Comparison
# ============================================================
def decode_config(cfg):
    """Split config into (decoder_label, encoder_label)."""
    parts   = cfg.split('_', 1)
    decoder = parts[0]
    encoder = parts[1] if len(parts) > 1 else ''
    # Order matters: longer names first to avoid substring collision
    _dec_map = {
        'deeplabv3plus': 'DeepLabV3+',
        'unetpp':        'UNet++',
        'upernet':       'UPerNet',
        'segformer':     'SegFormer',
        'unet':          'UNet',
    }
    dec_lbl = _dec_map.get(decoder, decoder)
    enc_lbl = encoder.replace('tu-', '').replace('_', ' ')
    return dec_lbl, enc_lbl

latex = []
latex.append(r'\begin{table*}[htbp]')
latex.append(r'\centering')
latex.append(
    r'\caption{Comparison of segmentation architectures on the Forest Inspection '
    r'dataset (test set: seq9). Results are reported as mean$\pm$std over three '
    r'independent runs with different random seeds.}'
)
latex.append(r'\label{tab:main_results}')
latex.append(r'\begin{tabular}{llcccccc}')
latex.append(r'\hline')
latex.append(
    r'\textbf{Decoder} & \textbf{Encoder} & '
    r'\textbf{Params(M)} & \textbf{FLOPs(G)} & \textbf{FPS} & '
    r'\textbf{mIoU(\%)} & \textbf{PA(\%)} & \textbf{mF1(\%)} \\'
)
latex.append(r'\hline')

prev_group = None
for cfg, row in mean_df.iterrows():
    grp = row['group']
    if grp != prev_group:
        # Group divider
        latex.append(f'\\multicolumn{{8}}{{l}}{{\\textit{{{grp}}}}} \\\\')
        latex.append(r'\hline')
        prev_group = grp
    s = std_df.loc[cfg]
    dec, enc = decode_config(cfg)
    miou_s = f"{row['mIoU']:.1f}$\\pm${s['mIoU']:.1f}"
    pa_s   = f"{row['PA']:.1f}$\\pm${s['PA']:.1f}"
    mf1_s  = f"{row['mF1']:.1f}$\\pm${s['mF1']:.1f}"
    fps_s  = f'{row["fps"]:.1f}' if row['fps'] > 0 else '—'
    flops_s= f'{row["flops_G"]:.1f}' if row['flops_G'] > 0 else '—'
    line   = (f'{dec} & {enc} & {row["params_M"]:.1f} & '
              f'{flops_s} & {fps_s} & {miou_s} & {pa_s} & {mf1_s} \\\\')
    latex.append(line)

latex.append(r'\hline')
latex.append(r'\end{tabular}')
latex.append(r'\end{table*}')

latex_str = '\n'.join(latex)
print(latex_str)
with open(OUT_DIR / 'table1_main.tex', 'w', encoding='utf-8') as f:
    f.write(latex_str)
print('\nSaved: table1_main.tex')

In [ ]:
# ============================================================
# LATEX TABLE 2: Per-class IoU — Top 5
# ============================================================
short_cls = ['Sky','Decid.','Conif.','Fallen','Dirt','GndVeg','Rocks','Build.','Fence','Car','Empty']

latex2 = []
latex2.append(r'\begin{table*}[htbp]')
latex2.append(r'\centering')
latex2.append(r'\caption{Per-class IoU (\%) of the top-5 configurations on the test set.}')
latex2.append(r'\label{tab:per_class}')
cols = 'l' + 'c' * len(short_cls)
latex2.append(f'\\begin{{tabular}}{{{cols}}}')
latex2.append(r'\hline')
latex2.append('\\textbf{Model} & ' +
              ' & '.join(f'\\textbf{{{c}}}' for c in short_cls) + r' \\')
latex2.append(r'\hline')

for cfg, row in mean_df.head(5).iterrows():
    dec, enc = decode_config(cfg)
    model_lbl = f'{dec}+{enc[:14]}'
    vals = []
    for c in CLASS_NAMES_FULL:
        v = row[f'iou_{c}']
        vals.append('—' if v < 0 else f'{v:.1f}')
    latex2.append(f'{model_lbl} & ' + ' & '.join(vals) + r' \\')

latex2.append(r'\hline')
latex2.append(r'\end{tabular}')
latex2.append(r'\end{table*}')

latex2_str = '\n'.join(latex2)
print(latex2_str)
with open(OUT_DIR / 'table2_per_class.tex', 'w', encoding='utf-8') as f:
    f.write(latex2_str)
print('\nSaved: table2_per_class.tex')

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================
print('=' * 65)
print('NB04 XONG!')
print('=' * 65)
outs = sorted(OUT_DIR.iterdir())
figs = [f for f in outs if f.suffix == '.png']
texs = [f for f in outs if f.suffix == '.tex']

print(f'\nFigures ({len(figs)}):')
for f in figs:
    print(f'  {f.name:45s} {f.stat().st_size/1024:.0f} KB')

print(f'\nLaTeX ({len(texs)}):')
for f in texs:
    print(f'  {f.name}')

best_cfg = mean_df.index[0]
print(f'\nBEST MODEL: {best_cfg}')
print(f'  mIoU   = {mean_df.loc[best_cfg, "mIoU"]:.1f} +/- {std_df.loc[best_cfg, "mIoU"]:.1f}%')
print(f'  PA     = {mean_df.loc[best_cfg, "PA"]:.1f} +/- {std_df.loc[best_cfg, "PA"]:.1f}%')
print(f'  mF1    = {mean_df.loc[best_cfg, "mF1"]:.1f} +/- {std_df.loc[best_cfg, "mF1"]:.1f}%')
print(f'  Params = {mean_df.loc[best_cfg, "params_M"]:.1f}M')
print(f'  FLOPs  = {mean_df.loc[best_cfg, "flops_G"]:.1f}G')
print(f'  FPS    = {mean_df.loc[best_cfg, "fps"]:.1f}')